# NB04 — Musanga satellite abundance and uncertainty mapping

**CURRENT WORKFLOW: run only Sections 06–09 below.** Sections 00–01 are archived legacy setup and are **not required** for the cached workflow. In particular, do not run the old SSD-staging cell. The experimental Blocks 02–14 remain accessible in Git history; this notebook now prioritizes prediction intervals, ensemble spread and area of applicability.

## 00 — USER SETTINGS

In [ ]:
# ==================== USER SETTINGS ====================
SCALES_M = [10,30,50,75,100,125]
FEATURE_SETS = ['ANNUAL_S2','EMBEDDING','S2_PLUS_EMBEDDING']
MODELS = ['RIDGE','RF','EXTRA_TREES','HIST_GB']
TARGETS = ['mean_probability','cover_fraction_p05']
MAX_SAMPLES_PER_MOSAIC = 8000
MIN_UAV_COVERAGE = .90
SEED = 20261022
COPY_UAV_TO_SSD = True
COPY_FEATURES_TO_SSD = True
FORCE_REBUILD = False  # keep False: resumes cached aggregate/sample/fold files
N_JOBS = 2
SAVE_FIGURES = True
# ========================================================

# BLOCK 06 — targeted high-cover recovery (independent of full Block 04 rerun)
B06_RUN = True
B06_SCALES_M = [75,100,125]
B06_FEATURE_SETS = ['EMBEDDING','S2_PLUS_EMBEDDING']
B06_TARGETS = ['mean_probability','cover_fraction_p05']
B06_WEIGHT_STRENGTHS = [0,2,5]  # 0=standard boosting; >0 emphasizes cover above 20%
B06_HIGH_THRESHOLD = .40
B06_REBUILD = False


# ========== BLOCKS 07–08: ROI seasonal LOSO settings ==========
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
FINAL=ROOT/"Final"
ROI_PATH=ROOT/"data/Orthoo_RGB/ROI.shp"
SCALES=[75,100,125]
TARGETS=["mean_probability"]  # optional: add "cover_fraction_p05"
FEATURE_SETS={
 "EMB":["EMB_ANNUAL"],
 "QUARTERLY":["S2_QUARTERLY_3YR"],
 "EMB_QUARTERLY":["EMB_ANNUAL","S2_QUARTERLY_3YR"],
 "EMB_ANNUAL_QUARTERLY":["EMB_ANNUAL","RAW_ANNUAL","S2_QUARTERLY_3YR"]
}
CV_SCHEMES=["FIVE_MOSAICS","FOUR_FORESTS"]
MAX_CELLS_PER_MOSAIC=8000
MIN_ROI_COVERAGE=0.90    # fractional pixel coverage inside ROI
ROI_SUBPIXELS=4          # 4x4 supersampling
MIN_UAV_COVERAGE=0.90
MIN_QUARTER_CLEAR=1
SEED=20261022
WEIGHT_STRENGTHS=[0,2]
HIGH_COVER_THRESHOLD=0.40
FORCE_SAMPLES=False
FORCE_FOLDS=False
SAVE_PLOTS=True


## 01 — Mount Drive, audit cached inputs and stage them on local SSD

In [ ]:
%pip -q install rasterio geopandas pandas pyarrow scikit-learn matplotlib scipy
import os, json, math, time, hashlib, shutil, gc, warnings
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from affine import Affine
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OLD=ROOT/'Outputs/V3/04_satellite_Musanga/features'
MAPS=ROOT/'Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
OUT=ROOT/'Outputs/V3_1/04_satellite_full_warmstart'
CACHE=OUT/'cache';AGG=CACHE/'aggregates';SAMPLE=CACHE/'samples';FOLDS=OUT/'folds';FIG=OUT/'figures'
for p in (CACHE,AGG,SAMPLE,FOLDS,FIG):p.mkdir(parents=True,exist_ok=True)
SSD=Path('/content/musanga_nb04_ssd');(SSD/'uav').mkdir(parents=True,exist_ok=True);(SSD/'features').mkdir(exist_ok=True)
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
SITE_MOSAICS={'Baego':['Baego.tif'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif','Mos_5_6_transparent_mosaic_group1_georef.tif'],
 'Yangambi':['Yangambi_georef.tif'],'Yoko':['Yoko_kvo_georef.tif']}
FAMILY_NAMES={'ANNUAL_S2':['RAW_ANNUAL'],'EMBEDDING':['EMB_ANNUAL'],
              'S2_PLUS_EMBEDDING':['RAW_ANNUAL','EMB_ANNUAL']}
def uavfile(name):return MAPS/f'{Path(name).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
def featfile(site,fam):return OLD/fam/f'{site}_{fam}_{YEARS[site]}_10m.tif'
def staged(path,folder,copy):
    assert path.is_file(),('Missing required cached file',path)
    target=folder/path.name
    if not copy:return path
    srcstat=path.stat()
    if not target.exists() or target.stat().st_size!=srcstat.st_size:
        print('COPY to SSD:',path.name,round(srcstat.st_size/1e9,2),'GB',flush=True)
        tmp=target.with_name(target.name+'.partial')
        shutil.copyfile(path,tmp);os.replace(tmp,target)
    return target
UAV={name:staged(uavfile(name),SSD/'uav',COPY_UAV_TO_SSD) for ns in SITE_MOSAICS.values() for name in ns}
FAMILY={'RAW_ANNUAL','EMB_ANNUAL'}
assert all(featfile(site,fam).is_file() for site in SITE_MOSAICS for fam in FAMILY)
FEATURES={(site,fam):staged(featfile(site,fam),SSD/'features',COPY_FEATURES_TO_SSD)
          for site in SITE_MOSAICS for fam in sorted(FAMILY)}
print('Input audit: ',len(UAV),'UAV maps,',len(FEATURES),'cached predictor stacks')
print('SSD free GB:',round(shutil.disk_usage(SSD).free/1e9,2),flush=True)


## Archived legacy setup ends here

The following independent mapping workflow uses existing `roi_cells_100m.parquet` / `roi_cells_75m.parquet` and aligned annual satellite raster grids. Do not run the legacy staging cells above.

## 06 — One configuration, no extraction

Run 06 → 09 only. Default: 100 m, annual embeddings + quarterly Sentinel-2. Balanced capped training prevents MOS from dominating; globally summarized geographic validation averages four forests equally. No legacy blocks are required.

In [ ]:
# 06 — INPUTS, settings, forest-balanced sampling & spatial partitions
# Run ONLY sections 06 onward. No raster extraction or legacy setup needed.
from pathlib import Path
import numpy as np, pandas as pd, rasterio, matplotlib.pyplot as plt, json
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import r2_score
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
CACHE=ROOT/"Final/Outputs/V3_1/04B_roi_quarterly_LOSO/samples"
SAT=ROOT/"Final/Outputs/V3/04_satellite_Musanga/features/RAW_ANNUAL"
OUT=ROOT/"Final/Outputs/V3_1/04_uncertainty_mapping"
OUT.mkdir(parents=True,exist_ok=True)
# Settings: change here; validation and map stages use the same values.
SCALE_M=100  # recommended; optionally rerun at 75 m for sensitivity
YEARS={"Baego":2023,"MOS":2024,"Yangambi":2024,"Yoko":2020}
FORESTS=["Baego","MOS","Yangambi","Yoko"]
FEATURE_FAMILIES=["EMB_ANNUAL","S2_QUARTERLY_3YR"]
N_ENSEMBLE=5
TRAIN_CAP_PER_FOREST=500  # MOS downsampled; other forests capped if larger
BLOCK_SIZE_M=500
SPATIAL_BUFFER_M=150
TEST_FRACTION=.20
CAL_FRACTION=.20
INTERVAL_LEVEL=.90
REPEATS=3
SEED=20261022
FORCE_VALIDATION=False
# Required artifact, aligned original Block 07 satellite/UAV sampling
SAMPLE_PATH=CACHE/f"roi_cells_{SCALE_M}m.parquet"
assert SAMPLE_PATH.is_file(),f"Cached data missing: {SAMPLE_PATH}"
all_cells=pd.read_parquet(SAMPLE_PATH).reset_index(drop=True)
assert set(all_cells.site.unique())==set(FORESTS),all_cells.site.unique()
assert {"site","group","mosaic","row","col","mean_probability"}.issubset(all_cells.columns)
FEATURE_COLS=[c for c in all_cells if any(c.startswith(f+"__") for f in FEATURE_FAMILIES)]
assert len(FEATURE_COLS)>60,"Missing embedding/seasonal feature columns"
assert all_cells[FEATURE_COLS].notna().all().all()
assert all_cells.mean_probability.between(0,1).all()
assert not all_cells.duplicated(["site","group","row","col"]).any(),"Duplicate spatial samples"
# A cell's row/col are its position on that forest's satellite grid (not geodetic coordinates).
# Group-prefixed spatial blocks prevent falsely merging the two MOS mosaics.
df=all_cells.copy()
df["x_m"]=(df.col.to_numpy(float)+.5)*SCALE_M
df["y_m"]=(df.row.to_numpy(float)+.5)*SCALE_M
df["block_id"]=(df.group.astype(str)+"_"+
    np.floor(df.x_m/BLOCK_SIZE_M).astype(int).astype(str)+"_"+
    np.floor(df.y_m/BLOCK_SIZE_M).astype(int).astype(str))
print("CACHED SPATIAL SUPPORT:",len(df),"cells at",SCALE_M,"m")
display(df.groupby("site").mean_probability.agg(
    n="size",mean_cover="mean",p90=lambda s:s.quantile(.9),
    high_fraction=lambda s:(s>=.4).mean()).round(3))
print("Full-region feature rasters beyond the UAV footprints are NOT available in this cache.")
def subset_cap(frame,max_n,rng):
    if len(frame)<=max_n:return frame.copy()
    return frame.iloc[np.sort(rng.choice(len(frame),max_n,replace=False))].copy()
def balanced_training(frame,rng):
    portions=[]
    for site in FORESTS:
        d=subset_cap(frame[frame.site==site],TRAIN_CAP_PER_FOREST,rng)
        if len(d)<30:raise ValueError(f"Too few training rows for {site}: {len(d)}")
        portions.append(d)
    return pd.concat(portions,ignore_index=True)
def split_spatial(frame,repeat):
    rng=np.random.default_rng(SEED+SCALE_M*100+repeat)
    tr_parts=[];ca_parts=[];te_parts=[]
    for site in FORESTS:
        whole=frame[frame.site==site].copy()
        groups=whole.groupby("block_id").indices
        keys=list(groups);rng.shuffle(keys)
        held={"test":[],"cal":[]};cursor=0
        for split,fraction in [("test",TEST_FRACTION),("cal",CAL_FRACTION)]:
            count=0
            while cursor<len(keys) and count<fraction*len(whole):
                key=keys[cursor];cursor+=1
                held[split].append(key);count+=len(groups[key])
        a=whole[~whole.block_id.isin(held["test"]+held["cal"])].copy()
        b=whole[whole.block_id.isin(held["cal"])].copy()
        c=whole[whole.block_id.isin(held["test"])].copy()
        # Real-world distances only within the same forest footprint.
        def remove_near(candidate,reference):
            if candidate.empty or reference.empty:return candidate.iloc[0:0].copy()
            nearest=cKDTree(reference[["x_m","y_m"]].to_numpy()).query(
                candidate[["x_m","y_m"]].to_numpy(),k=1)[0]
            return candidate.iloc[np.flatnonzero(nearest>=SPATIAL_BUFFER_M)].copy()
        b=remove_near(b,a);c=remove_near(c,a);c=remove_near(c,b)
        if min(len(a),len(b),len(c))<8:
            raise ValueError(("Insufficient independent blocks",site,repeat,
                              len(a),len(b),len(c),"Try smaller buffer or larger coverage"))
        tr_parts.append(a);ca_parts.append(b);te_parts.append(c)
    return tuple(pd.concat(items,ignore_index=True) for items in (tr_parts,ca_parts,te_parts))
def fit_ensemble(train,predict_sets,seed):
    models=[];rng=np.random.default_rng(seed)
    for i in range(N_ENSEMBLE):
        fit=balanced_training(train,np.random.default_rng(seed+i*113))
        # Per-tree random forest cap varies across seeds/blocks; this is sample/subsample uncertainty.
        model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
            max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
            early_stopping=False,random_state=seed+i)
        model.fit(fit[FEATURE_COLS].to_numpy(np.float32),
                  fit.mean_probability.to_numpy(np.float32),
                  sample_weight=fit.site.map(fit.site.value_counts().rdiv(1)).to_numpy(float)
                    / np.mean(fit.site.map(fit.site.value_counts().rdiv(1)).to_numpy(float)))
        models.append(model)
    outputs=[]
    for group in predict_sets:
        x=group[FEATURE_COLS].to_numpy(np.float32)
        arr=np.stack([np.clip(m.predict(x),0,1) for m in models],axis=0)
        outputs.append((arr.mean(axis=0),arr.std(axis=0)))
    return models,outputs


## 07 — Forest-balanced spatial validation and asymmetric prediction interval calibration

Separate spatial training/calibration/test blocks. Calibration uses signed error quantiles, with three **predicted-abundance** strata (0–10%, 10–25%, ≥25%), pooling low-support strata. Earlier symmetric calibration outputs remain archived under their original names; v2 caches are distinct. Inspect 08 before GIS export.

In [ ]:
# 07 — SPATIALLY RESERVED TRAIN / CALIBRATION / TEST & global, equal-forest metrics
VALID_DIR=OUT/"validation";VALID_DIR.mkdir(exist_ok=True)
def empirical_quantile(values,level):
    v=np.sort(np.asarray(values,float))
    k=min(len(v),int(np.ceil((len(v)+1)*level)))
    return float(v[k-1])
def forest_balanced_calibration(cal,pred):
    # Calibrate *signed* residual tails. Equal-forest subsample prevents MOS
    # setting the error distribution; predicted bins make interval shape adaptive.
    # These are empirical, not coverage-guaranteed under spatial/domain shift.
    rng=np.random.default_rng(SEED+824)
    n_each=int(cal.site.value_counts().min())
    idx=np.concatenate([rng.choice(np.flatnonzero(cal.site.to_numpy()==forest),
                       n_each,replace=False) for forest in FORESTS])
    error=cal.mean_probability.to_numpy(float)[idx]-np.asarray(pred,float)[idx]
    fitted=np.asarray(pred,float)[idx]
    alpha=1-INTERVAL_LEVEL
    def signed_limits(v):
        v=np.sort(np.asarray(v,float));n=len(v)
        # Conservative signed-residual empirical quantiles.
        lo=v[max(0,int(np.floor((n+1)*alpha/2))-1)]
        hi=v[min(n-1,int(np.ceil((n+1)*(1-alpha/2)))-1)]
        return [float(lo),float(hi)]
    global_limits=signed_limits(error)
    # Bins depend ONLY on predicted fractions; sparse strata use pooled limits.
    edges=[0,.10,.25,1.000001]
    bins=[]
    for a,b in zip(edges[:-1],edges[1:]):
        within=(fitted>=a)&(fitted<b)
        n=int(within.sum())
        limits=signed_limits(error[within]) if n>=25 else global_limits
        bins.append(dict(min_pred=a,max_pred=b,n_cal=n,limits=limits,
                         pooled_fallback=n<25))
    return dict(n_per_forest=n_each,n_total=len(idx),global_limits=global_limits,
                bins=bins,bin_edges=edges)
def interval_bounds(pred,calibration):
    pred=np.asarray(pred,float)
    low=np.empty_like(pred);high=np.empty_like(pred)
    for segment in calibration["bins"]:
        mask=(pred>=segment["min_pred"])&(pred<segment["max_pred"])
        lo,hi=segment["limits"]
        low[mask]=np.clip(pred[mask]+lo,0,1)
        high[mask]=np.clip(pred[mask]+hi,0,1)
    assert np.all(high>=low),"Asymmetric interval boundaries invalid"
    return low,high
def score(sub):
    y=sub.observed.to_numpy(float);p=sub.predicted.to_numpy(float)
    high=y>=.4
    return dict(n=len(sub),R2=r2_score(y,p) if len(y)>1 and y.std()>1e-8 else np.nan,
                MAE_pp=100*np.mean(abs(y-p)),
                RMSE_pp=100*np.sqrt(np.mean((y-p)**2)),
                bias_pp=100*np.mean(p-y),
                PI90_coverage=np.mean((y>=sub.lower90)&(y<=sub.upper90)),
                PI90_width_pp=100*np.mean(sub.upper90-sub.lower90),
                high_n=int(high.sum()),
                high_bias_pp=100*np.mean((p-y)[high]) if high.any() else np.nan,
                high_PI90_coverage=np.mean(
                    (y[high]>=sub.lower90.to_numpy()[high])&
                    (y[high]<=sub.upper90.to_numpy()[high])) if high.any() else np.nan,
                ensemble_sd_pp=100*np.mean(sub.ensemble_sd))
def novelty_fit(tr,query):
    # PCA nearest-neighbour support check: exploratory, not a validated AOA guarantee.
    trX=tr[FEATURE_COLS].to_numpy(np.float32)
    qX=query[FEATURE_COLS].to_numpy(np.float32)
    scaler=StandardScaler().fit(trX)
    z=scaler.transform(trX);q=scaler.transform(qX)
    pca=PCA(n_components=min(10,z.shape[1],len(z)-1),random_state=SEED).fit(z)
    z=pca.transform(z);q=pca.transform(q)
    nn=NearestNeighbors(n_neighbors=2).fit(z)
    training_distance=nn.kneighbors(z)[0][:,1]
    test_distance=nn.kneighbors(q,n_neighbors=1)[0][:,0]
    limit=float(np.quantile(training_distance,.95))
    return test_distance,limit
all_results=[];all_predictions=[];all_cal=[]
for repeat in range(REPEATS):
    cache=VALID_DIR/f"heldout_asymmetric_v2_{SCALE_M}m_rep{repeat}.parquet"
    calibration_meta=VALID_DIR/f"calibration_asymmetric_v2_{SCALE_M}m_rep{repeat}.json"
    if cache.exists() and calibration_meta.exists() and not FORCE_VALIDATION:
        test_out=pd.read_parquet(cache)
        info=json.loads(calibration_meta.read_text())
        print("CACHED VALIDATION",repeat,flush=True)
    else:
        train,cal,test=split_spatial(df,repeat)
        models,preds=fit_ensemble(train,[cal,test],SEED+repeat*193)
        cal_mean,_=preds[0];test_mean,test_sd=preds[1]
        calibration=forest_balanced_calibration(cal,cal_mean)
        lower,upper=interval_bounds(test_mean,calibration)
        dist,limit=novelty_fit(train,test)
        test_out=test[["site","group","row","col"]].copy()
        test_out["observed"]=test.mean_probability.to_numpy(float)
        test_out["predicted"]=test_mean
        test_out["lower90"]=lower
        test_out["upper90"]=upper
        test_out["ensemble_sd"]=test_sd
        test_out["novelty_distance"]=dist
        test_out["novel"]=dist>limit
        info=dict(repeat=repeat,n_train=len(train),n_cal=len(cal),n_test=len(test),
                  n_cal_each_forest=calibration["n_per_forest"],
                  calibration=calibration,novelty_limit=limit)
        test_out.to_parquet(cache,index=False)
        calibration_meta.write_text(json.dumps(info,indent=2))
        print("VALIDATED",repeat,info,flush=True)
    all_predictions.append(test_out)
    all_cal.append(info)
    for forest,chunk in test_out.groupby("site"):
        all_results.append(dict(repeat=repeat,forest=forest,**score(chunk)))
metrics=pd.DataFrame(all_results)
metrics.to_csv(VALID_DIR/f"per_forest_audit_asymmetric_v2_{SCALE_M}m.csv",index=False)
# GLOBAL metrics ONLY; equal weight to forests and equal weight to spatial repetitions.
numerical=["MAE_pp","RMSE_pp","bias_pp","PI90_coverage","PI90_width_pp",
           "high_PI90_coverage","ensemble_sd_pp"]
summary=metrics.groupby("repeat")[numerical].mean(numeric_only=True)
# R² needs a global pooled definition, not averaging per-forest R².
for rep,pred in enumerate(all_predictions):
    y=pred.observed.to_numpy(float);p=pred.predicted.to_numpy(float)
    forest=pred.site.to_numpy()
    w=np.array([1/(len(FORESTS)*np.sum(forest==s)) for s in forest])
    mu=np.dot(w,y);total=np.dot(w,(y-mu)**2)
    summary.loc[rep,"balanced_global_R2"]=1-np.dot(w,(y-p)**2)/total
    summary.loc[rep,"n_test"]=len(pred)
    summary.loc[rep,"n_test_high"]=int(np.sum(y>=.4))
summary.to_csv(VALID_DIR/f"global_metrics_by_repeat_asymmetric_v2_{SCALE_M}m.csv")
global_means=summary.mean(numeric_only=True).to_frame("mean").join(
    summary.std(numeric_only=True).to_frame("sd"))
global_means.to_csv(VALID_DIR/f"global_metrics_asymmetric_v2_{SCALE_M}m.csv")
print("GLOBAL EQUAL-FOREST SPATIAL VALIDATION (mean and SD across repeats)")
display(global_means.round(3))
print("Calibration metadata:",all_cal)
print("Spatial test predictions and per-forest audits saved; global metrics are primary.")


## 08 — Global uncertainty diagnostics

Inspect abundance-dependent interval coverage, ensemble disagreement, and held-out prediction calibration. This is the decision gate before regional deployment.

In [ ]:
# 08 — Ensemble, interval and predictor-domain diagnostics (cached held-out predictions)
from IPython.display import display
P=pd.concat(all_predictions,ignore_index=True)
P["error_pp"]=100*(P.predicted-P.observed)
P["observed_class"]=pd.cut(P.observed,[-.001,.2,.4,1.001],
                            labels=["<20%","20–40%","≥40%"])
bins=P.groupby("observed_class",observed=True).agg(
    n=("observed","size"),MAE_pp=("error_pp",lambda v:abs(v).mean()),
    bias_pp=("error_pp","mean"),
    PI90_coverage=("observed",lambda s:np.nan))
# Calculate coverage from the original frame, not from separate columns in .agg.
coverage=P.assign(covered=(P.observed>=P.lower90)&(P.observed<=P.upper90))
bins["PI90_coverage"]=coverage.groupby("observed_class",observed=True).covered.mean()
bins.to_csv(OUT/f"uncertainty_by_abundance_{SCALE_M}m.csv")
print("GLOBAL diagnostic only; repeated test sets are not independent replicates")
display(bins.round(3))
fig,axes=plt.subplots(1,3,figsize=(13,3.8),constrained_layout=True)
axes[0].scatter(P.observed*100,P.predicted*100,s=6,alpha=.15,rasterized=True)
axes[0].plot([0,100],[0,100],"--",color="grey")
axes[0].set(xlim=(0,100),ylim=(0,100),xlabel="UAV target (%)",
            ylabel="Predicted Musanga (%)",title="Spatially held-out predictions")
axes[1].hist(P.ensemble_sd*100,bins=30)
axes[1].set(xlabel="Ensemble SD (percentage points)",title="Model disagreement")
axes[2].scatter(P.predicted*100,100*(P.upper90-P.lower90),
                s=6,alpha=.2,rasterized=True)
axes[2].set(xlabel="Predicted Musanga (%)",ylabel="90% interval width (pp)",
            title="Empirically calibrated interval width")
fig.savefig(OUT/f"global_diagnostics_{SCALE_M}m.png",dpi=175)
plt.show();plt.close(fig)


## 09 — Export provisional UAV-footprint GIS products

Uses the calibrated asymmetric intervals from Block 07 alongside predicted abundance, ensemble spread, and predictor novelty. Validation and mapping do **not** establish coverage across the unsampled Kisangani region. Do not export until the held-out high-cover coverage is acceptable.

In [ ]:
# 09 — Fit final balanced ensemble and map cached UAV-mosaic footprints.
# GeoTIFF bands: predicted fraction, lower90, upper90, ensemble_sd, novelty_distance, outside_domain.
# This is NOT a seamless region-wide Kisangani map; unobserved/non-cached pixels are nodata.
from affine import Affine
from rasterio.transform import Affine as RasterAffine
from rasterio.crs import CRS
MAPPING=OUT/f"maps_{SCALE_M}m";MAPPING.mkdir(exist_ok=True)
final_models,[(mean_pred,sd_pred)]=fit_ensemble(df,[df],SEED+901)
dist,limit=novelty_fit(balanced_training(df,np.random.default_rng(SEED+900)),
                      df)
# Independent holdout-based interval radii; pooled for mapping as a transparent
# pilot approximation. Calibration refers to the re-fitted full-data model only approximately.
# Aggregate the calibration residual offsets across spatial repeats by prediction bin.
# Median limits are descriptive for final refit; validate coverage again before deployment.
calibration_for_map={"bins":[]}
for i in range(len(all_cal[0]["calibration"]["bins"])):
    template=all_cal[0]["calibration"]["bins"][i]
    limits=np.median(np.array([m["calibration"]["bins"][i]["limits"] for m in all_cal]),axis=0)
    calibration_for_map["bins"].append(dict(min_pred=template["min_pred"],
        max_pred=template["max_pred"],limits=limits.tolist()))
final=df[["site","group","mosaic","row","col","mean_probability"]].copy()
final["predicted"]=mean_pred
final["lower90"],final["upper90"]=interval_bounds(mean_pred,calibration_for_map)
final["ensemble_sd"]=sd_pred
final["novelty_distance"]=dist
final["outside_domain"]=(dist>limit).astype("float32")
final.to_parquet(MAPPING/"sampled_mosaic_predictions.parquet",index=False)
for forest in FORESTS:
    path=SAT/f"{forest}_RAW_ANNUAL_{YEARS[forest]}_10m.tif"
    assert path.is_file(),f"Missing source-grid definition: {path}"
    with rasterio.open(path) as ds:
        fac=SCALE_M/10
        tr=ds.transform*RasterAffine.scale(fac,fac)
        height=int(np.ceil(ds.height/fac));width=int(np.ceil(ds.width/fac))
        profile=dict(driver="GTiff",width=width,height=height,count=6,
                     dtype="float32",crs=ds.crs,transform=tr,
                     nodata=-9999.,compress="deflate",tiled=True)
        part=final[final.site==forest]
        for mosaic,sub in part.groupby("group"):
            bands=np.full((6,height,width),-9999.,np.float32)
            rr=sub.row.to_numpy(int);cc=sub.col.to_numpy(int)
            assert (rr>=0).all() and (rr<height).all() and (cc>=0).all() and (cc<width).all()
            fields=["predicted","lower90","upper90","ensemble_sd",
                    "novelty_distance","outside_domain"]
            for i,field in enumerate(fields):
                bands[i,rr,cc]=sub[field].to_numpy("float32")
            dest=MAPPING/f"{mosaic}_{SCALE_M}m_pred_PI90_ensemble_AOA.tif"
            with rasterio.open(dest,"w",**profile) as out:
                out.write(bands)
                for i,field in enumerate(fields,1):out.set_band_description(i,field)
            print("WROTE",dest,"eligible cells",len(sub),flush=True)
manifest=dict(resolution_m=SCALE_M,features=FEATURE_FAMILIES,
    forests=FORESTS,ensemble_members=N_ENSEMBLE,
    training_cap_per_forest=TRAIN_CAP_PER_FOREST,
    spatial_calibration_limits=[m["calibration"] for m in all_cal],
    mapping_asymmetric_calibration=calibration_for_map,
    novelty_threshold=limit,
    provenance="Cached Block 07 ROI samples only; NOT a full regional raster",
    warning="Intervals are empirical pilots; full model retrained on all UAV samples, regional validity not established.")
(MAPPING/"README_mapping_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Completed ROI-footprint mapping. For full Kisangani coverage, first extend matched annual embeddings and quarterly Sentinel-2 predictors to the regional grid.")
